# XAI-MedCrossNet — Max-AUC Pipeline (VinDr-Mammo)
Multi-backbone (DINO ViT-B/16 + ConvNeXt-V2 + SwinV2) + Cross-Attention Fusion + Breast ROI Crop + Focal Loss + Patient-Aware GroupKFold + TTA + 5-Fold Ensemble + Isotonic Calibration + Full Evaluation (ROC, Confusion Matrix, Calibration Curve)

**Note:** current published SOTA on VinDr-Mammo malignancy/BI-RADS classification is ~0.85 AUC. This pipeline maximizes legitimate techniques toward 0.86-0.89; 0.90+ is a stretch outcome, not guaranteed.

In [ ]:
!pip install -q timm albumentations opencv-python-headless scikit-learn torch torchvision

In [ ]:
import os, gc, cv2, timm, random, warnings, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, recall_score,
                              precision_score, accuracy_score, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report,
                              precision_recall_curve, average_precision_score, brier_score_loss)
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

## Config

In [ ]:
SEED = 42
IMG_DIR = Path("./images_png")
BREAST_CSV = Path("breast-level_annotations.csv")
FINDING_CSV = Path("finding_annotations.csv")

IMG_SIZE = 384
BATCH_SIZE = 16          # RTX 5060 (~8GB): 16 @384 / 8 @512. RTX 5070 (~12GB): 24-32 @384 / 12 @512.
ACCUM_STEPS = 2
EPOCHS = 25
N_FOLDS = 5
EMBED_DIM = 256
DROPOUT_P = 0.4
AUX_WEIGHT = 0.3
LR = 1e-4
WEIGHT_DECAY = 1e-5
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0
N_TTA = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BACKBONES = {
    "dino_vit": "vit_base_patch16_224.dino",
    "convnext_v2": "convnextv2_base.fcmae_ft_in22k_in1k",
    "swin_v2": "swinv2_base_window8_256.ms_in22k_ft_in1k",
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Load & Merge Annotations

In [ ]:
def load_dataframe():
    df_raw = pd.read_csv(BREAST_CSV)
    malignant_birads = {"BI-RADS 3", "BI-RADS 4", "BI-RADS 5", "3", "4", "5"}
    df_raw["target_label"] = df_raw["breast_birads"].isin(malignant_birads).astype(int)
    df_raw["is_R"] = (df_raw["laterality"] == "R").astype(int)
    density_map = {f"DENSITY {c}": i for i, c in enumerate(["A", "B", "C", "D"])}
    df_raw["density_encoded"] = df_raw["breast_density"].map(density_map).fillna(1).astype(int)
    df_raw["patient_id"] = df_raw["study_id"]

    cc = df_raw[df_raw.view_position == "CC"].drop_duplicates(["study_id", "laterality"])
    mlo = df_raw[df_raw.view_position == "MLO"].drop_duplicates(["study_id", "laterality"])
    df = cc.merge(mlo, on=["study_id", "laterality"], suffixes=("_cc", "_mlo"))
    df["target_label"] = df["target_label_cc"]
    df["is_R"] = df["is_R_cc"]
    df["density_encoded"] = df["density_encoded_cc"]
    df["patient_id"] = df["patient_id_cc"]
    df["split"] = df["split_cc"]
    return df.reset_index(drop=True)

df = load_dataframe()
print(f"Total breast-pairs: {len(df):,} | Unique patients: {df.patient_id.nunique():,}")
print(df["target_label"].value_counts(normalize=True))

## Exploratory Class Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df["target_label"].value_counts().plot(kind="bar", ax=ax[0], color=["#4C72B0", "#DD8452"])
ax[0].set_xticklabels(["Benign (0)", "Malignant (1)"], rotation=0)
ax[0].set_title("Class distribution")

df["density_encoded"].value_counts().sort_index().plot(kind="bar", ax=ax[1], color="#55A868")
ax[1].set_xticklabels(["A", "B", "C", "D"], rotation=0)
ax[1].set_title("Breast density distribution")
plt.tight_layout()
plt.show()

## Breast ROI Crop (background removal)

In [ ]:
def crop_breast_roi(img, pad_frac=0.02):
    if img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    H, W = gray.shape[:2]
    pad_x, pad_y = int(w * pad_frac), int(h * pad_frac)
    x0, y0 = max(0, x - pad_x), max(0, y - pad_y)
    x1, y1 = min(W, x + w + pad_x), min(H, y + h + pad_y)
    if (x1 - x0) < 50 or (y1 - y0) < 50:
        return img
    return img[y0:y1, x0:x1]

In [ ]:
sample_row = df.iloc[0]
sample_path = IMG_DIR / f"{sample_row['image_id_cc']}.png"
if sample_path.exists():
    orig = cv2.imread(str(sample_path), cv2.IMREAD_COLOR)
    cropped = crop_breast_roi(orig)
    fig, ax = plt.subplots(1, 2, figsize=(9, 5))
    ax[0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)); ax[1].set_title("ROI Cropped"); ax[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Sample image not found at expected path — check IMG_DIR.")

## Dataset & Transforms

In [ ]:
class VinDrMultiViewDataset(Dataset):
    def __init__(self, df, img_dir, tab_matrix, aux_target, transform=None, roi_crop=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.tab_mat = np.nan_to_num(tab_matrix.astype(np.float32), nan=0.0)
        self.aux_target = np.nan_to_num(np.asarray(aux_target, dtype=np.float32), nan=0.0)
        self.transform = transform
        self.roi_crop = roi_crop

    def __len__(self):
        return len(self.df)

    def _load(self, image_id):
        p = self.img_dir / f"{image_id}.png"
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.roi_crop:
            img = crop_breast_roi(img)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc_img = self._load(row["image_id_cc"])
        mlo_img = self._load(row["image_id_mlo"])
        if self.transform:
            cc_img = self.transform(image=cc_img)["image"]
            mlo_img = self.transform(image=mlo_img)["image"]
        return {
            "cc_img": cc_img,
            "mlo_img": mlo_img,
            "tabular": torch.tensor(self.tab_mat[idx], dtype=torch.float32),
            "aux_target": torch.tensor(self.aux_target[idx], dtype=torch.float32),
            "label": torch.tensor(row["target_label"], dtype=torch.float32),
        }

In [ ]:
def get_transforms(img_size):
    norm = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    train_tf = A.Compose([
        A.Resize(img_size, img_size),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Affine(rotate=(-8, 8), translate_percent=(0.0, 0.05), scale=(0.95, 1.05), p=0.3),
        norm, ToTensorV2(),
    ])
    val_tf = A.Compose([A.Resize(img_size, img_size), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0), norm, ToTensorV2()])
    tta_tf = [
        val_tf,
        A.Compose([A.Resize(img_size, img_size), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0), A.HorizontalFlip(p=1.0), norm, ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0), A.Affine(rotate=(3, 3), p=1.0), norm, ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0), A.Affine(rotate=(-3, -3), p=1.0), norm, ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0), A.VerticalFlip(p=1.0), norm, ToTensorV2()]),
    ]
    return train_tf, val_tf, tta_tf[:N_TTA]

train_tf, val_tf, tta_tf = get_transforms(IMG_SIZE)

## Model — Cross-Attention Fusion over Multi-Backbone Features

In [ ]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, img_feat, tab_feat):
        Q = img_feat.unsqueeze(1)
        K = tab_feat.unsqueeze(1)
        attn_out, _ = self.attn(Q, K, K)
        return self.norm(img_feat + self.dropout(attn_out.squeeze(1)))


class XAIMedCrossNet(nn.Module):
    def __init__(self, backbone_name, tab_dim, embed_dim=EMBED_DIM, dropout=DROPOUT_P):
        super().__init__()
        self.cc_backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.mlo_backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.cc_backbone.num_features

        self.cc_proj = nn.Linear(feat_dim, embed_dim)
        self.mlo_proj = nn.Linear(feat_dim, embed_dim)
        self.tab_proj = nn.Sequential(nn.Linear(tab_dim, embed_dim), nn.ReLU(), nn.Dropout(dropout))

        self.view_cross_attn = MultiHeadCrossAttention(embed_dim, num_heads=4, dropout=dropout)
        self.tab_cross_attn = MultiHeadCrossAttention(embed_dim, num_heads=4, dropout=dropout)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )
        self.aux_head = nn.Sequential(nn.Linear(embed_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, cc_img, mlo_img, tab):
        cc_feat = self.cc_proj(self.cc_backbone(cc_img))
        mlo_feat = self.mlo_proj(self.mlo_backbone(mlo_img))
        tab_feat = self.tab_proj(tab)

        fused_view = self.view_cross_attn(cc_feat, mlo_feat)
        fused_tab = self.tab_cross_attn(fused_view, tab_feat)

        combined = torch.cat([fused_view, fused_tab], dim=1)
        logit = self.classifier(combined).squeeze(1)
        aux_logit = self.aux_head(fused_view).squeeze(1)
        return logit, aux_logit

## Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        loss = self.alpha * (1 - p_t) ** self.gamma * bce
        return loss.mean()

## Train / Evaluate / TTA Helpers

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        cc = batch["cc_img"].to(device)
        mlo = batch["mlo_img"].to(device)
        tab = batch["tabular"].to(device)
        with autocast():
            logit, _ = model(cc, mlo, tab)
        preds.append(torch.sigmoid(logit).float().cpu().numpy())
        labels.append(batch["label"].numpy())
    return np.concatenate(preds), np.concatenate(labels)


@torch.no_grad()
def predict_tta(model, df_subset, tab_mat, aux_target, tta_transforms, device, batch_size=BATCH_SIZE):
    model.eval()
    all_preds = []
    for tf in tta_transforms:
        ds = VinDrMultiViewDataset(df_subset, IMG_DIR, tab_mat, aux_target, transform=tf, roi_crop=True)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
        preds = []
        for batch in loader:
            cc = batch["cc_img"].to(device)
            mlo = batch["mlo_img"].to(device)
            tab = batch["tabular"].to(device)
            with autocast():
                logit, _ = model(cc, mlo, tab)
            preds.append(torch.sigmoid(logit).float().cpu().numpy())
        all_preds.append(np.concatenate(preds))
    return np.mean(all_preds, axis=0)


def train_one_fold(model, train_loader, val_loader, device, epochs=EPOCHS):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()
    cls_loss_fn = FocalLoss()
    aux_loss_fn = nn.MSELoss()

    best_auc, best_state = 0.0, None
    history = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        for i, batch in enumerate(train_loader):
            cc = batch["cc_img"].to(device)
            mlo = batch["mlo_img"].to(device)
            tab = batch["tabular"].to(device)
            aux_t = batch["aux_target"].to(device)
            label = batch["label"].to(device)

            with autocast():
                logit, aux_logit = model(cc, mlo, tab)
                loss = cls_loss_fn(logit, label) + AUX_WEIGHT * aux_loss_fn(aux_logit, aux_t)
                loss = loss / ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += loss.item() * ACCUM_STEPS
            if (i + 1) % ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        scheduler.step()
        val_preds, val_labels = evaluate(model, val_loader, device)
        val_auc = roc_auc_score(val_labels, val_preds)
        history.append({"epoch": epoch + 1, "train_loss": running_loss / len(train_loader), "val_auc": val_auc})
        print(f"  epoch {epoch+1}/{epochs} | train_loss={running_loss/len(train_loader):.4f} | val AUC = {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        gc.collect()
        torch.cuda.empty_cache()

    model.load_state_dict(best_state)
    return model, best_auc, history

## Main Training Loop — 5-Fold x 3-Backbone

In [ ]:
tab_cols = ["is_R"]
scaler_tab = StandardScaler()
tab_matrix_all = scaler_tab.fit_transform(df[tab_cols].values)
aux_target_all = df["density_encoded"].values

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_models = {name: [] for name in BACKBONES}
oof_backbone_preds = {name: np.zeros(len(df)) for name in BACKBONES}
oof_labels = np.zeros(len(df))
all_history = []

for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(df, df["target_label"], df["patient_id"])):
    print(f"\n===== FOLD {fold+1}/{N_FOLDS} =====")
    tr_df, vl_df = df.iloc[tr_idx].reset_index(drop=True), df.iloc[vl_idx].reset_index(drop=True)
    tab_tr, tab_vl = tab_matrix_all[tr_idx], tab_matrix_all[vl_idx]
    aux_tr, aux_vl = aux_target_all[tr_idx], aux_target_all[vl_idx]
    oof_labels[vl_idx] = vl_df["target_label"].values

    train_ds = VinDrMultiViewDataset(tr_df, IMG_DIR, tab_tr, aux_tr, transform=train_tf)
    val_ds = VinDrMultiViewDataset(vl_df, IMG_DIR, tab_vl, aux_vl, transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    for name, backbone_id in BACKBONES.items():
        print(f"-- training backbone: {name} --")
        model = XAIMedCrossNet(backbone_id, tab_dim=len(tab_cols)).to(DEVICE)
        model, best_auc, history = train_one_fold(model, train_loader, val_loader, DEVICE)
        for h in history:
            h.update({"fold": fold + 1, "backbone": name})
        all_history.extend(history)
        print(f"   fold {fold+1} {name} best AUC = {best_auc:.4f}")

        tta_preds = predict_tta(model, vl_df, tab_vl, aux_vl, tta_tf, DEVICE)
        oof_backbone_preds[name][vl_idx] = tta_preds

        ckpt_path = f"fold{fold+1}_{name}.pt"
        torch.save(model.state_dict(), ckpt_path)
        fold_models[name].append(ckpt_path)

        del model
        gc.collect()
        torch.cuda.empty_cache()

## Training Curves (per backbone)

In [ ]:
hist_df = pd.DataFrame(all_history)
fig, axes = plt.subplots(1, len(BACKBONES), figsize=(15, 4), sharey=True)
for ax, name in zip(axes, BACKBONES):
    sub = hist_df[hist_df.backbone == name]
    for f in sub.fold.unique():
        fs = sub[sub.fold == f]
        ax.plot(fs.epoch, fs.val_auc, label=f"fold {f}")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val AUC")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Per-Backbone OOF AUC + Weighted Ensemble

In [ ]:
weights = {}
for name in BACKBONES:
    auc = roc_auc_score(oof_labels, oof_backbone_preds[name])
    weights[name] = auc
    print(f"[{name}] OOF AUC = {auc:.4f}")

total_w = sum(weights.values())
weights = {k: v / total_w for k, v in weights.items()}
ensemble_oof = sum(weights[name] * oof_backbone_preds[name] for name in BACKBONES)
ensemble_auc = roc_auc_score(oof_labels, ensemble_oof)
print(f"\n[ENSEMBLE] weighted OOF AUC = {ensemble_auc:.4f}")
print(f"weights = {weights}")

## Isotonic Calibration + Optimal Threshold

In [ ]:
iso = IsotonicRegression(out_of_bounds="clip")
calibrated_oof = iso.fit_transform(ensemble_oof, oof_labels)

best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(oof_labels, (calibrated_oof >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

sens_thresh, sens_prec = 0.5, 0.0
for t in np.arange(0.9, 0.05, -0.01):
    y_pred = (calibrated_oof >= t).astype(int)
    sens = recall_score(oof_labels, y_pred, zero_division=0)
    if sens >= 0.80:
        sens_thresh = t
        break

print(f"F1-optimal threshold = {best_thresh:.2f} | F1 = {best_f1:.4f}")
print(f"High-sensitivity (>=0.80 recall) threshold = {sens_thresh:.2f}")

brier = brier_score_loss(oof_labels, calibrated_oof)
print(f"Brier score (calibrated) = {brier:.4f}")

## Confusion Matrix

In [ ]:
y_pred_f1 = (calibrated_oof >= best_thresh).astype(int)
y_pred_sens = (calibrated_oof >= sens_thresh).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, y_pred, title in zip(axes, [y_pred_f1, y_pred_sens], ["F1-optimal threshold", "High-sensitivity threshold"]):
    cm = confusion_matrix(oof_labels, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

print("Classification report (F1-optimal threshold):")
print(classification_report(oof_labels, y_pred_f1, target_names=["Benign", "Malignant"]))

## ROC Curve & Precision-Recall Curve

In [ ]:
fpr, tpr, _ = roc_curve(oof_labels, calibrated_oof)
precision, recall, _ = precision_recall_curve(oof_labels, calibrated_oof)
ap = average_precision_score(oof_labels, calibrated_oof)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, label=f"AUC = {ensemble_auc:.4f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AP = {ap:.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()
plt.tight_layout()
plt.show()

## Calibration Curve

In [ ]:
prob_true, prob_pred = calibration_curve(oof_labels, calibrated_oof, n_bins=10)
plt.figure(figsize=(5, 5))
plt.plot(prob_pred, prob_true, marker="o", label="Calibrated model")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title(f"Calibration curve (Brier = {brier:.4f})")
plt.legend()
plt.tight_layout()
plt.show()

## Per-Backbone Comparison Table

In [ ]:
summary_rows = []
for name in BACKBONES:
    auc = roc_auc_score(oof_labels, oof_backbone_preds[name])
    preds_bin = (oof_backbone_preds[name] >= 0.5).astype(int)
    summary_rows.append({
        "Backbone": name,
        "AUC": auc,
        "Accuracy": accuracy_score(oof_labels, preds_bin),
        "Precision": precision_score(oof_labels, preds_bin, zero_division=0),
        "Recall": recall_score(oof_labels, preds_bin, zero_division=0),
        "F1": f1_score(oof_labels, preds_bin, zero_division=0),
    })
summary_rows.append({
    "Backbone": "ENSEMBLE (weighted + calibrated)",
    "AUC": ensemble_auc,
    "Accuracy": accuracy_score(oof_labels, y_pred_f1),
    "Precision": precision_score(oof_labels, y_pred_f1, zero_division=0),
    "Recall": recall_score(oof_labels, y_pred_f1, zero_division=0),
    "F1": f1_score(oof_labels, y_pred_f1, zero_division=0),
})
summary_df = pd.DataFrame(summary_rows)
summary_df

## Save Artifacts

In [ ]:
np.save("oof_ensemble_preds.npy", ensemble_oof)
np.save("oof_calibrated_preds.npy", calibrated_oof)
np.save("oof_labels.npy", oof_labels)
summary_df.to_csv("backbone_comparison_summary.csv", index=False)
import joblib
joblib.dump(iso, "isotonic_calibrator.pkl")
joblib.dump(weights, "ensemble_weights.pkl")
print("Saved: oof_ensemble_preds.npy, oof_calibrated_preds.npy, oof_labels.npy, backbone_comparison_summary.csv, isotonic_calibrator.pkl, ensemble_weights.pkl")
print("Saved per-fold-per-backbone checkpoints: fold{1-5}_{backbone}.pt")

## Notes
- Published SOTA on VinDr-Mammo malignancy/BI-RADS classification is ~0.85 AUC (as of literature review, 2023-2026). This pipeline (ROI crop + 3-backbone ensemble + TTA + focal loss + calibration) targets 0.86-0.89 as a realistic ceiling; 0.90+ is possible but not guaranteed — if observed, check patient-level leakage between folds before trusting the number.
- Adjust `IMG_SIZE` / `BATCH_SIZE` based on actual GPU VRAM (RTX 5060 vs 5070) to avoid OOM.
- To reduce total training time, drop one backbone from `BACKBONES` for a 2-backbone ensemble first, then add the third if time allows.